# Region 6 - Mayassi-style Swiss-roll unrolling method audit

This notebook reproduces, explains, and audits the Xenium unrolling strategy described by Mayassi et al. (Nature 2024). It is intentionally organized as short, inspectable steps in the style of `B2_Region6_primary_479.ipynb`.

**Purpose:** determine whether the published manual-trace plus global-radius projection produces trustworthy unrolled coordinates for Region 6. This is a feasibility and method-QC notebook, not a biological hypothesis test.

**Important:** `roll_arc_fraction` is anatomically unoriented. It must not be called proximal-distal until the physical rolling direction is confirmed.


## Result in one sentence

The smooth-muscle/serosal trace can be reconstructed, but the published global-radius constraint is considered **not analysis-ready** for this irregular roll when it causes substantial assignment changes, failed projections, or endpoint pile-up. The evidence is calculated below rather than assumed.


In [1]:
suppressPackageStartupMessages({
  library(SeuratObject)
  library(ggplot2)
  library(dplyr)
  library(patchwork)
})
options(stringsAsFactors = FALSE, repr.plot.width = 12, repr.plot.height = 8)
set.seed(20260910)
cat('R version:', R.version.string, '
')


R version: R version 4.6.1 (2026-06-24 ucrt) 


In [2]:
PROJECT_ROOT <- Sys.getenv(
  'COLON_PROJECT_ROOT',
  unset = if (.Platform$OS.type == 'windows') {
    'D:/Xiaonan/CODEX_projects/Yanan_Xenium'
  } else {
    '/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu/YNH_Xenium'
  }
)
ANALYSIS_ROOT <- file.path(PROJECT_ROOT, 'colon_analysis')
PIPELINE_REPO <- file.path(ANALYSIS_ROOT, 'YNH_Xenium_Colon')
HPC_RETURN <- file.path(ANALYSIS_ROOT, 'HPC_return', '20260910_HPC_return')
QC_RDS <- file.path(HPC_RETURN, 'colon_qc_outputs', 'colon_qc_hpc', 'sections', 'Region_6', 'Region_6.phase0_2_qc.rds')
ANNOTATED_RDS <- file.path(HPC_RETURN, 'colon_downstream_outputs', 'full_notebook_qc_v2', '03_Region_6_Primary479', 'Region_6_spatial_passQC.rds')
TRACE_TSV <- file.path(ANALYSIS_ROOT, 'mayassi_unrolling_spike', 'Region_6', 'Region_6_trace_control_points.tsv')
OUT_DIR <- file.path(ANALYSIS_ROOT, 'mayassi_unrolling_notebook', 'Region_6')
MAX_CELLS <- as.integer(Sys.getenv('COLON_UNROLL_MAX_CELLS', unset = '12000'))
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
print(data.frame(name = c('PROJECT_ROOT','PIPELINE_REPO','QC_RDS','ANNOTATED_RDS','TRACE_TSV','OUT_DIR'), path = c(PROJECT_ROOT,PIPELINE_REPO,QC_RDS,ANNOTATED_RDS,TRACE_TSV,OUT_DIR)))


           name
1  PROJECT_ROOT
2 PIPELINE_REPO
3        QC_RDS
4 ANNOTATED_RDS
5     TRACE_TSV
6       OUT_DIR
                                                                                                                                                                                     path
1                                                                                                                                               D:\\Xiaonan\\CODEX_projects\\Yanan_Xenium
2                                                                                                               D:\\Xiaonan\\CODEX_projects\\Yanan_Xenium/colon_analysis/YNH_Xenium_Colon
3                        D:\\Xiaonan\\CODEX_projects\\Yanan_Xenium/colon_analysis/HPC_return/20260910_HPC_return/colon_qc_outputs/colon_qc_hpc/sections/Region_6/Region_6.phase0_2_qc.rds
4 D:\\Xiaonan\\CODEX_projects\\Yanan_Xenium/colon_analysis/HPC_return/20260910_HPC_return/colon_downstream_outputs/full_notebook_qc_v2/03_Region

In [3]:
if (.Platform$OS.type == 'windows' && grepl('^[Cc]:', normalizePath(tempdir(), winslash = '/'))) {
  stop('R temporary files are on C:. Set TMPDIR, TEMP, and TMP below colon_analysis/tmp.')
}
required_paths <- c(PIPELINE_REPO, QC_RDS, ANNOTATED_RDS, TRACE_TSV)
if (any(!file.exists(required_paths))) stop('Missing required path(s): ', paste(required_paths[!file.exists(required_paths)], collapse = '; '))
if (!is.finite(MAX_CELLS) || MAX_CELLS < 100L) stop('MAX_CELLS must be at least 100')
source(file.path(PIPELINE_REPO, 'R', 'source.R'))
cat('Environment and path checks: PASS
')


Environment and path checks: PASS


## Step 1 - Load both Region 6 objects and identify the analysis populations

Two returned objects are required:

1. The phase-0/2 QC bundle contains all 138,752 cells and the authoritative fixed `primary_include` mask.
2. The downstream Seurat object contains annotations but was destructively restricted in the B2 notebook using `primary_include_revised`.

The approved primary rule is exactly `5 < nFeature_Xenium < 200` and `10 < nCount_Xenium < 1000`. Segmentation and control flags are sensitivity annotations and must not redefine this mask.


In [4]:
qc_bundle <- readRDS(QC_RDS)
annotated <- readRDS(ANNOTATED_RDS)
qc_cells <- qc_bundle$cells
annotated_meta <- annotated[[]]
annotated_meta$cell_id <- extract_cell_ids(annotated_meta)
print(data.frame(object = c('QC bundle','annotated Seurat'), cells = c(nrow(qc_cells), nrow(annotated_meta)), columns = c(ncol(qc_cells), ncol(annotated_meta))))


            object  cells columns
1        QC bundle 138752      34
2 annotated Seurat 127935     241


In [5]:
primary_audit <- audit_fixed_primary_include(qc_cells)
print(primary_audit)


  n_cells n_fixed_primary n_stored_primary n_disagree stored_matches_fixed_rule
1  138752          133195           133195          0                      TRUE


In [6]:
fixed_primary <-
  qc_cells$nFeature_Xenium > 5 & qc_cells$nFeature_Xenium < 200 &
  qc_cells$nCount_Xenium > 10 & qc_cells$nCount_Xenium < 1000
revised_sensitivity <-
  (qc_cells$qc_core_pass %in% TRUE) &
  !(qc_cells$high_control_flag %in% TRUE) &
  !(qc_cells$segmentation_multiplet_flag %in% TRUE)
population_audit <- data.frame(
  population = c('all segmented cells','approved fixed primary','B2 revised sensitivity','annotated Seurat returned'),
  n_cells = c(nrow(qc_cells), sum(fixed_primary), sum(revised_sensitivity), nrow(annotated_meta))
)
population_audit$percent_of_all <- 100 * population_audit$n_cells / nrow(qc_cells)
print(population_audit)


                 population n_cells percent_of_all
1       all segmented cells  138752      100.00000
2    approved fixed primary  133195       95.99501
3    B2 revised sensitivity  127935       92.20408
4 annotated Seurat returned  127935       92.20408


In [7]:
annotated_ids <- annotated_meta$cell_id
population_contract <- data.frame(
  check = c('stored primary equals fixed formula','annotated IDs equal B2 revised sensitivity','annotated IDs equal approved primary','approved primary cells omitted from annotated object'),
  value = c(
    identical(qc_cells$primary_include %in% TRUE, fixed_primary),
    setequal(annotated_ids, qc_cells$cell_id[revised_sensitivity]),
    setequal(annotated_ids, qc_cells$cell_id[fixed_primary]),
    sum(fixed_primary & !revised_sensitivity)
  )
)
print(population_contract)


                                                 check value
1                  stored primary equals fixed formula     1
2           annotated IDs equal B2 revised sensitivity     1
3                 annotated IDs equal approved primary     0
4 approved primary cells omitted from annotated object  5260


### Statistical correction

The B2 filtering step is retained as a **sensitivity population**, not renamed as primary QC. The unrolling test sample below is drawn from all cells passing the fixed primary rule. This preserves the pre-specified inclusion definition and prevents post-QC morphology flags from silently changing the estimand.


## Step 2 - Define exactly what "smooth muscle" means here

Cell type is not an input to the projection equation. It is used only to guide and validate the manually drawn serosal/muscle trace.

Two non-equivalent definitions are available:

- **Marker-cluster definition:** every cell in a cluster whose highest mean marker score was `Smooth_muscle`. The five markers were `Tagln`, `Myh11`, `Cnn1`, `Myl9`, and `Des`.
- **Mayassi-reference definition:** cell-level Seurat label transfer from the Mayassi colon scRNA-seq reference, stored as `RefAll_subtype_predicted.id`.

The continuous `SubtypeScore_Smooth_muscle` is displayed because a trace should follow spatial muscle-layer signal rather than depend on one hard label.


In [8]:
smooth_marker_definition <- data.frame(
  Gene_Symbol = c('Tagln','Myh11','Cnn1','Myl9','Des'),
  CellType_main = 'Mural',
  CellType_subtype = 'Smooth_muscle'
)
smooth_marker_coverage <- calculate_marker_definition_coverage(
  smooth_marker_definition,
  rownames(annotated)
)
print(smooth_marker_coverage)


  CellType_main CellType_subtype n_defined n_present coverage
1         Mural    Smooth_muscle         5         5        1
                  genes_present genes_absent
1 Tagln, Myh11, Cnn1, Myl9, Des             


In [9]:
marker_smooth <- as.character(annotated_meta$Xenium_cluster_subtype) == 'Smooth_muscle'
reference_smooth <- as.character(annotated_meta$RefAll_subtype_predicted.id) == 'Smooth muscle'
definition_agreement <- compare_binary_cell_definitions(
  marker_smooth,
  reference_smooth,
  first_name = 'cluster-level five-marker definition',
  second_name = 'Mayassi reference-transfer definition'
)
print(definition_agreement)


                      first_definition                     second_definition
1 cluster-level five-marker definition Mayassi reference-transfer definition
  n_evaluated n_first n_second n_intersection n_union   jaccard
1      127935   12335     8059           7045   13349 0.5277549
  first_supported_by_second second_supported_by_first
1                  0.571139                 0.8741779


In [10]:
cluster_definition_audit <- data.frame(
  cluster = as.character(annotated_meta$cluster_res_0_8),
  assigned = as.character(annotated_meta$Xenium_cluster_subtype),
  second = as.character(annotated_meta$Xenium_cluster_subtype_second)
) %>%
  filter(assigned == 'Smooth_muscle') %>%
  count(cluster, assigned, second, name = 'n_cells')
print(cluster_definition_audit)


  cluster      assigned            second n_cells
1       2 Smooth_muscle Contractile_mural   12335


In [11]:
definition_plot_data <- annotated_meta %>%
  transmute(
    x = x_centroid,
    y = y_centroid,
    marker_cluster = marker_smooth,
    reference_transfer = reference_smooth,
    continuous_score = SubtypeScore_Smooth_muscle
  )
p_marker <- ggplot(definition_plot_data, aes(x, y, color = marker_cluster)) + geom_point(size = 0.05, alpha = 0.35) + coord_equal() + scale_color_manual(values = c('FALSE'='grey85','TRUE'='black')) + theme_void() + ggtitle('Cluster-level five-marker label')
p_reference <- ggplot(definition_plot_data, aes(x, y, color = reference_transfer)) + geom_point(size = 0.05, alpha = 0.35) + coord_equal() + scale_color_manual(values = c('FALSE'='grey85','TRUE'='#2166AC')) + theme_void() + ggtitle('Mayassi reference-transfer label')
score_limits <- quantile(definition_plot_data$continuous_score, c(0.50, 0.995), na.rm = TRUE)
p_score <- ggplot(definition_plot_data, aes(x, y, color = continuous_score)) + geom_point(size = 0.05, alpha = 0.45) + coord_equal() + scale_color_viridis_c(option = 'magma', limits = score_limits, oob = scales::squish) + theme_void() + ggtitle('Continuous smooth-muscle score')
celltype_plot <- p_marker + p_reference + p_score + plot_layout(ncol = 3)
ggsave(file.path(OUT_DIR, 'Region_6_smooth_muscle_definition_comparison.png'), celltype_plot, width = 18, height = 6, dpi = 200, bg = 'white')
celltype_plot


![Comparison of the two binary smooth-muscle definitions and the continuous score](../../mayassi_unrolling_notebook/Region_6/Region_6_smooth_muscle_definition_comparison.png)

The binary definitions are reported as descriptive agreement only. No cell-level significance test is valid because nearby Xenium cells are spatially correlated and are not biological replicates.


## Step 3 - Construct the manual serosal/muscle trace

Following the authors' Xenium workflow, an ordered path is drawn from the outer free edge toward the central hook. Here, the manual action is stored as an auditable TSV of ordered control points rather than hidden in an edited PNG.

Consecutive control points are linearly interpolated every 20 um. Cumulative Euclidean distance along this ordered line gives `roll_arc_length`; dividing by total length gives `roll_arc_fraction` from 0 to 1.


In [12]:
control_points <- read.delim(TRACE_TSV, check.names = FALSE)
control_points <- control_points[order(control_points$point_order), ]
trace <- interpolate_trace_control_points(control_points[, c('x','y')], spacing = 20)
trace <- add_trace_arc_length(trace)
trace_center <- c(x = mean(trace$x), y = mean(trace$y))
trace$radius_from_center <- sqrt((trace$x - trace_center[['x']])^2 + (trace$y - trace_center[['y']])^2)
trace_quality <- assess_trace_quality(trace)
print(data.frame(control_points = nrow(control_points), dense_points = nrow(trace), arc_length_um = max(trace$roll_arc_length), continuous = trace_quality$continuous, large_jumps = trace_quality$n_large_jumps))


  control_points dense_points arc_length_um continuous large_jumps
1             39         1251      24978.31       TRUE           0


In [13]:
trace_plot <- ggplot(definition_plot_data, aes(x, y, color = continuous_score)) +
  geom_point(size = 0.05, alpha = 0.40) +
  geom_path(data = trace, aes(x, y), inherit.aes = FALSE, color = '#00A651', linewidth = 0.55) +
  geom_point(data = control_points, aes(x, y), inherit.aes = FALSE, color = '#D73027', size = 0.65) +
  coord_equal() +
  scale_color_viridis_c(option = 'magma', limits = score_limits, oob = scales::squish) +
  theme_void() +
  ggtitle('Region 6 manual trace on continuous smooth-muscle score', subtitle = 'green: dense trace; red: ordered control points')
ggsave(file.path(OUT_DIR, 'Region_6_trace_on_smooth_score.png'), trace_plot, width = 9, height = 9, dpi = 220, bg = 'white')
trace_plot


![Manual trace over the continuous smooth-muscle score](../../mayassi_unrolling_notebook/Region_6/Region_6_trace_on_smooth_score.png)

Visual review is indispensable: a mathematically smooth trace can still be biologically wrong if it crosses unsupported tissue or jumps between coils.


## Step 4 - Reproduce the Mayassi projection

For each cell, Euclidean distance is calculated to every dense trace point. The published constraint computes a single global center as the mean trace position and excludes trace points whose center-distance is greater than the cell's center-distance. The nearest remaining point is accepted.

The longitudinal coordinate is the accepted point's cumulative arc distance. The wall-depth coordinate is the unsigned cell-to-trace distance in micrometres. An unconstrained nearest-trace projection is also calculated strictly as a diagnostic; disagreement quantifies sensitivity to the published radial rule.


In [14]:
primary_cells <- qc_cells[fixed_primary, , drop = FALSE]
primary_coordinates <- data.frame(
  cell_id = as.character(primary_cells$cell_id),
  x = as.numeric(primary_cells$x_centroid),
  y = as.numeric(primary_cells$y_centroid)
)
set.seed(20260910)
selected <- if (nrow(primary_coordinates) > MAX_CELLS) sort(sample.int(nrow(primary_coordinates), MAX_CELLS)) else seq_len(nrow(primary_coordinates))
cells_pilot <- primary_coordinates[selected, , drop = FALSE]
annotation_match <- match(cells_pilot$cell_id, annotated_meta$cell_id)
cells_pilot$smooth_muscle_marker <- marker_smooth[annotation_match]
cells_pilot$smooth_muscle_reference <- reference_smooth[annotation_match]
print(data.frame(approved_primary_cells = nrow(primary_coordinates), pilot_cells = nrow(cells_pilot), pilot_with_annotation = sum(!is.na(annotation_match))))


  approved_primary_cells pilot_cells pilot_with_annotation
1                 133195       12000                 11527


In [15]:
projected <- project_cells_to_trace(
  cells_pilot,
  trace,
  inward_constraint = TRUE,
  center = trace_center
)
cat('Published constrained projection complete for', nrow(projected), 'cells
')


Published constrained projection complete for 12000 cells


In [16]:
unconstrained <- project_cells_to_trace(
  cells_pilot,
  trace,
  inward_constraint = FALSE,
  center = trace_center
)
projected$unconstrained_trace_index <- unconstrained$trace_index
projected$unconstrained_roll_arc_fraction <- unconstrained$roll_arc_fraction
projected$unconstrained_wall_distance <- unconstrained$wall_distance
projected$projection_changed_by_constraint <- projected$trace_index != projected$unconstrained_trace_index
cat('Unconstrained diagnostic projection complete
')


Unconstrained diagnostic projection complete


## Step 5 - Quantify whether the transformation is trustworthy

These are descriptive geometry/QC summaries, not inferential statistics. Pre-specified warning gates for this feasibility notebook are:

- projection success below 95%;
- more than 20% of assignments changed by the global-radius constraint;
- more than 5% of assigned cells falling in the first or last 1% of the trace.

The thresholds are engineering review gates, not biological significance cut-offs.


In [17]:
valid <- !is.na(projected$trace_index)
valid_unconstrained <- !is.na(projected$unconstrained_trace_index)
endpoint <- projected$roll_arc_fraction <= 0.01 | projected$roll_arc_fraction >= 0.99
endpoint_unconstrained <- projected$unconstrained_roll_arc_fraction <= 0.01 | projected$unconstrained_roll_arc_fraction >= 0.99
metrics <- data.frame(
  metric = c('approved_primary_cells','pilot_cells','projection_success_fraction','constraint_changed_fraction','constrained_wall_distance_median_um','constrained_wall_distance_p95_um','unconstrained_wall_distance_median_um','unconstrained_wall_distance_p95_um','constrained_endpoint_fraction','unconstrained_endpoint_fraction'),
  value = c(
    nrow(primary_coordinates), nrow(projected), mean(valid),
    mean(projected$projection_changed_by_constraint[valid], na.rm = TRUE),
    median(projected$wall_distance[valid], na.rm = TRUE),
    as.numeric(quantile(projected$wall_distance[valid], 0.95, na.rm = TRUE)),
    median(projected$unconstrained_wall_distance[valid_unconstrained], na.rm = TRUE),
    as.numeric(quantile(projected$unconstrained_wall_distance[valid_unconstrained], 0.95, na.rm = TRUE)),
    mean(endpoint[valid], na.rm = TRUE), mean(endpoint_unconstrained[valid_unconstrained], na.rm = TRUE)
  )
)
print(metrics)


                                  metric        value
1                 approved_primary_cells 1.331950e+05
2                            pilot_cells 1.200000e+04
3            projection_success_fraction 9.124167e-01
4            constraint_changed_fraction 6.971413e-01
5    constrained_wall_distance_median_um 4.947042e+02
6       constrained_wall_distance_p95_um 1.829298e+03
7  unconstrained_wall_distance_median_um 2.155277e+02
8     unconstrained_wall_distance_p95_um 8.309387e+02
9          constrained_endpoint_fraction 2.974701e-01
10       unconstrained_endpoint_fraction 8.900000e-02


In [18]:
smooth_distance <- data.frame(
  definition = c('marker-cluster smooth muscle','all other annotated cells'),
  n = c(sum(valid & projected$smooth_muscle_marker %in% TRUE), sum(valid & projected$smooth_muscle_marker %in% FALSE)),
  median_wall_distance_um = c(
    median(projected$wall_distance[valid & projected$smooth_muscle_marker %in% TRUE], na.rm = TRUE),
    median(projected$wall_distance[valid & projected$smooth_muscle_marker %in% FALSE], na.rm = TRUE)
  )
)
print(smooth_distance)


                    definition    n median_wall_distance_um
1 marker-cluster smooth muscle 1033                123.7060
2    all other annotated cells 9486                521.6797


In [19]:
write_tsv_gz(projected, file.path(OUT_DIR, 'Region_6_primary_unrolled_pilot_cells.tsv.gz'))
write.table(trace, file.path(OUT_DIR, 'Region_6_ordered_dense_trace.tsv'), sep = '	', quote = FALSE, row.names = FALSE)
write.table(metrics, file.path(OUT_DIR, 'Region_6_method_audit_metrics.tsv'), sep = '	', quote = FALSE, row.names = FALSE)
write.table(definition_agreement, file.path(OUT_DIR, 'Region_6_smooth_muscle_definition_agreement.tsv'), sep = '	', quote = FALSE, row.names = FALSE)
cat('Audit tables written to:', OUT_DIR, '
')


Audit tables written to: D:\Xiaonan\CODEX_projects\Yanan_Xenium/colon_analysis/mayassi_unrolling_notebook/Region_6 


In [20]:
rolled_plot <- ggplot(projected[valid, ], aes(x, y, color = roll_arc_fraction)) +
  geom_point(size = 0.16, alpha = 0.65) +
  geom_path(data = trace, aes(x, y), inherit.aes = FALSE, color = 'black', linewidth = 0.3) +
  coord_equal() + scale_color_viridis_c(option = 'turbo', name = 'roll arc
fraction') +
  theme_void() + ggtitle('Published constrained projection on rolled coordinates')
ggsave(file.path(OUT_DIR, 'Region_6_primary_rolled_arc.png'), rolled_plot, width = 9, height = 9, dpi = 220, bg = 'white')
rolled_plot


![Rolled coordinates coloured by the constrained longitudinal coordinate](../../mayassi_unrolling_notebook/Region_6/Region_6_primary_rolled_arc.png)


In [21]:
p_constrained <- ggplot(projected[valid, ], aes(roll_arc_fraction, wall_distance)) + geom_point(size = 0.18, alpha = 0.35, color = '#B2182B') + theme_bw() + labs(x = 'Constrained roll arc fraction', y = 'Wall distance (um)', title = 'Published global-radius constraint')
p_unconstrained <- ggplot(projected[valid_unconstrained, ], aes(unconstrained_roll_arc_fraction, unconstrained_wall_distance)) + geom_point(size = 0.18, alpha = 0.35, color = '#2166AC') + theme_bw() + labs(x = 'Unconstrained roll arc fraction', y = 'Nearest-trace distance (um)', title = 'Unconstrained diagnostic')
unrolled_comparison <- p_constrained + p_unconstrained
ggsave(file.path(OUT_DIR, 'Region_6_unrolled_projection_comparison.png'), unrolled_comparison, width = 14, height = 6, dpi = 220, bg = 'white')
unrolled_comparison


![Constrained and unconstrained unrolled-coordinate diagnostics](../../mayassi_unrolling_notebook/Region_6/Region_6_unrolled_projection_comparison.png)


In [22]:
agreement_plot_data <- projected[valid & valid_unconstrained, ]
constraint_plot <- ggplot(agreement_plot_data, aes(unconstrained_roll_arc_fraction, roll_arc_fraction)) +
  geom_point(size = 0.18, alpha = 0.35) + geom_abline(slope = 1, intercept = 0, color = '#D73027') +
  coord_equal(xlim = c(0,1), ylim = c(0,1)) + theme_bw() +
  labs(x = 'Unconstrained arc fraction', y = 'Constrained arc fraction', title = 'Sensitivity to the global-radius rule')
ggsave(file.path(OUT_DIR, 'Region_6_constraint_assignment_comparison.png'), constraint_plot, width = 7, height = 7, dpi = 220, bg = 'white')
constraint_plot


![Cell assignments with and without the global-radius constraint](../../mayassi_unrolling_notebook/Region_6/Region_6_constraint_assignment_comparison.png)


## Step 6 - Statistical and scientific interpretation

- Cells are spatial observations, not independent biological replicates; therefore this notebook does not attach cell-level p-values to marker/reference agreement or wall-distance differences.
- The manual trace is a geometric annotation and must be reviewed visually. Optimizing the trace to improve metrics while crossing unsupported tissue is invalid.
- The marker-cluster label is assigned at cluster level and must not be described as an independently validated single-cell identity.
- Reference transfer labels require score/confidence review and are not a ground truth.
- A high constraint-change fraction or endpoint pile-up indicates model misspecification, not a biological gradient.
- The current longitudinal coordinate remains unoriented until the rolling direction is documented.


In [23]:
metric_value <- setNames(metrics$value, metrics$metric)
method_gates <- data.frame(
  diagnostic = c('projection success >= 0.95','constraint-changed fraction <= 0.20','endpoint fraction <= 0.05'),
  observed = c(metric_value[['projection_success_fraction']], metric_value[['constraint_changed_fraction']], metric_value[['constrained_endpoint_fraction']]),
  pass = c(metric_value[['projection_success_fraction']] >= 0.95, metric_value[['constraint_changed_fraction']] <= 0.20, metric_value[['constrained_endpoint_fraction']] <= 0.05)
)
method_status <- if (all(method_gates$pass)) 'PASS_FOR_SCALE_UP' else 'NEEDS_REVISION_DO_NOT_SCALE'
print(method_gates)
cat('Method status:', method_status, '
')


                           diagnostic  observed  pass
1          projection success >= 0.95 0.9124167 FALSE
2 constraint-changed fraction <= 0.20 0.6971413 FALSE
3           endpoint fraction <= 0.05 0.2974701 FALSE
Method status: NEEDS_REVISION_DO_NOT_SCALE 


## Final decision

If any gate above fails, the current Mayassi global-radius implementation must remain a feasibility result. Do not propagate it to Regions 1-5 and do not use its arc coordinate for eosinophil or other biological inference.

The appropriate next method comparison is a topology-aware local projection that retains the same manually reviewed trace while reducing cross-coil ambiguity. Morphology images and physical proximal-distal orientation remain valuable external validation evidence.


In [24]:
sessionInfo()


R version 4.6.1 (2026-06-24 ucrt)
Platform: x86_64-w64-mingw32/x64
Running under: Windows 11 x64 (build 22631)

Matrix products: default
  LAPACK version 3.12.1

locale:
[1] C
system code page: 65001

time zone: Asia/Shanghai
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] patchwork_1.3.2    dplyr_1.2.1        ggplot2_4.0.3      SeuratObject_5.4.0
[5] sp_2.2-3           jsonlite_2.0.0    

loaded via a namespace (and not attached):
 [1] Matrix_1.7-5        gtable_0.3.6        future.apply_1.20.2
 [4] compiler_4.6.1      tidyselect_1.2.1    Rcpp_1.1.2         
 [7] parallel_4.6.1      textshaping_1.0.5   systemfonts_1.3.2  
[10] globals_0.19.1      scales_1.4.0        lattice_0.22-9     
[13] R6_2.6.1            labeling_0.4.3      generics_0.1.4     
[16] dotCall64_1.2       future_1.75.0       tibble_3.3.1       
[19] pillar_1.11.1       RColorBrewer_1.1-3  rlang_1.3.0        
[22]